# Interpretação de Domínio dos Arquivos AQG

O [notebook 1](aqg_context_analysis.ipynb) motivou uma memória intermediária pelo custo de contexto. O [notebook 2](aqg_memory_sketch.ipynb) construiu registros, relações e referências para a fonte, preservando o AQG. Este notebook acrescenta a segunda etapa da C1: **atribuir significado a um conjunto pequeno de dados**.

O recorte inclui cinco indicadores declarados, unidades por torre e candidatos para a relação entre descritivos e famílias instanciadas. Os objetos continuam sendo a estrutura da memória; valores interpretados ficam associados a eles, com estado, unidade, regra e procedência.

```text
C0  AQG original + hash
 |           |
 |           +-> fonte dos valores e evidências
 v
C1  registros preservados -> conceitos e cálculos de domínio
                              |
                              +-> indicadores da solução
                              +-> unidades por torre instanciada
                              +-> relações candidatas dos descritivos
```

Não há chamadas a LLM. As visões para o agente (C2) e a memória de trabalho (C3) ficam para os próximos experimentos.

## 1. Setup e Acervo

In [1]:
import hashlib
import json
import math
import sys
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 75)

REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/cad/model.py").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [2]:
from experiments.aqg_memory import (
    MISSING,
    SourceRef,
    build_memory,
    child_path,
    lookup,
)

DATA_DIR = REPO_ROOT / "data/aqg"
PATHS = sorted(DATA_DIR.rglob("*.json"))
if not PATHS:
    raise FileNotFoundError(f"Nenhum AQG encontrado em {DATA_DIR}.")

SOURCES, MEMORIES = {}, {}
for path in PATHS:
    name = str(path.relative_to(DATA_DIR))
    raw = path.read_bytes()
    project = json.loads(raw)
    snapshot = hashlib.sha256(raw).hexdigest()
    SOURCES[name] = project
    MEMORIES[name] = build_memory(project, name, snapshot)

print(f"{len(MEMORIES)} AQGs carregados com a mesma construção da C1 do notebook 2.")

37 AQGs carregados com a mesma construção da C1 do notebook 2.


## 2. Contrato Semântico Inicial

| Conceito | Fonte em `solution.outputs` | Tipo aceito | Unidade do contrato |
|---|---|---|---|
| Total de torres | `TotalDeTorres.value` | inteiro não negativo | torre |
| Total de unidades | `TotalDeUnidades.value` | inteiro não negativo | unidade habitacional |
| Vagas projetadas | `NumeroDeVagasProjetado.value` | inteiro não negativo | vaga |
| Área total construída | `AreaTotalConstruida.value` | número finito não negativo | m² |
| CA projetado | `CAProjetado.value` | número finito não negativo | adimensional |

Esses valores são **declarados pelo exportador**, não recalculados nem certificados como corretos. A unidade é uma convenção explícita deste protótipo, coerente com o uso no projeto, e não um metadado lido de cada indicador. A confirmação formal com o produtor do AQG ainda é necessária. Não há conversão por ordem de grandeza, nem substituição por campos com nomes semelhantes.

Cada leitura distingue `known`, `missing`, `null`, `invalid` e `ambiguous`. Zero é um valor válido. Strings numéricas e booleanos não são convertidos. Duas ocorrências do mesmo indicador são ambíguas, mesmo quando têm valores iguais. Para uma ausência, a evidência aponta ao registro onde a leitura foi tentada.

In [3]:
State = Literal["known", "missing", "null", "invalid", "ambiguous"]


@dataclass(frozen=True)
class Reading:
    value: int | float | None
    state: State
    unit: str
    nature: Literal["declared", "derived"]
    sources: tuple[SourceRef, ...]
    rule: str


CONTRACT = {
    "total_torres": ("TotalDeTorres", "torre", True),
    "total_unidades": ("TotalDeUnidades", "unidade habitacional", True),
    "vagas_projetadas": ("NumeroDeVagasProjetado", "vaga", True),
    "area_construida": ("AreaTotalConstruida", "m²", False),
    "ca_projetado": ("CAProjetado", "adimensional", False),
}
UNIT_BASIS = "Convenção do protótipo; validar com o produtor do AQG."


def number(record, pointer, unit, integer=True):
    value = lookup(record.data, pointer)
    state = "missing" if value is MISSING else "null" if value is None else "known"
    if state == "known":
        valid = (
            type(value) is int
            if integer
            else type(value) in (int, float)
            and (type(value) is int or math.isfinite(value))
        )
        if not valid or value < 0:
            state = "invalid"
    source = SourceRef(record.source.snapshot, record.source.pointer + pointer)
    return Reading(
        value if state == "known" else None,
        state,
        unit,
        "declared",
        (record.source if state == "missing" else source,),
        "scalar/v1",
    )


def declared_indicators(memory):
    result = {}
    for concept, (kind, unit, integer) in CONTRACT.items():
        matches = [o for o in memory.solution.outputs if o.data.get("type") == kind]
        if len(matches) == 1:
            result[concept] = number(matches[0], "/value", unit, integer)
        else:
            result[concept] = Reading(
                None,
                "ambiguous" if matches else "missing",
                unit,
                "declared",
                tuple(o.source for o in matches) or (memory.solution.record.source,),
                f"unique-output/{kind}/v1",
            )
    return result

## 3. Unidades das Torres

Uma torre é uma instância cuja família resolvida tem tipo `Tower` ou `DynamicTower3d`. A regra segue a interpretação existente em `src/cad/model.py::tower_units`:

```text
unidades por pavimento = soma das contagens em tipologias
unidades da torre     = unidades por pavimento × n_pavimentos_unidades
```

Lemos `properties.tipologias`; quando ausente ou `null`, tentamos `properties.pav_tipo.tipologias`. Se ambos existirem, o campo direto tem precedência, como na aplicação. As contagens e o número de pavimentos devem ser inteiros não negativos; um mapa vazio não autoriza assumir zero. A hipótese é que essas tipologias se repetem em todos os pavimentos de unidades. Não extraímos pavimentos da string de apresentação do descritivo.

A derivação registra tanto os operandos quanto as evidências do vínculo entre instância e família. Instâncias sem família resolvida permanecem explícitas e impedem uma conclusão de completude. Os identificadores continuam restritos ao snapshot, sem associação entre versões ou soluções.

In [4]:
TOWER_KINDS = {"Tower", "DynamicTower3d"}


def derived(value, state, unit, sources, rule):
    return Reading(value, state, unit, "derived", tuple(dict.fromkeys(sources)), rule)


def family_units(family):
    record = family.record
    path = "/properties/tipologias"
    if lookup(record.data, path) is MISSING or lookup(record.data, path) is None:
        path = "/properties/pav_tipo/tipologias"
    typologies = lookup(record.data, path)
    ref = SourceRef(record.source.snapshot, record.source.pointer + path)
    if isinstance(typologies, dict) and typologies:
        parts = [
            number(record, child_path(path, key), "unidade habitacional/pavimento")
            for key in typologies
        ]
        state = next((p.state for p in parts if p.state != "known"), "known")
        per_floor = derived(
            sum(p.value for p in parts) if state == "known" else None,
            state,
            "unidade habitacional/pavimento",
            [s for p in parts for s in p.sources],
            "sum-typologies/v1",
        )
    else:
        state = (
            "missing"
            if typologies is MISSING
            else "null"
            if typologies is None
            else "invalid"
        )
        per_floor = derived(
            None,
            state,
            "unidade habitacional/pavimento",
            [record.source if state == "missing" else ref],
            "sum-typologies/v1",
        )
    floors = number(record, "/properties/n_pavimentos_unidades", "pavimento")
    state = per_floor.state if per_floor.state != "known" else floors.state
    total = derived(
        per_floor.value * floors.value if state == "known" else None,
        state,
        "unidade habitacional",
        per_floor.sources + floors.sources,
        "units-per-floor-times-floors/v1",
    )
    return per_floor, total


def tower_semantics(memory):
    towers, unresolved = {}, []
    for pid, placement in memory.solution.placements.items():
        if placement.family is None:
            continue
        if placement.family.status != "resolved":
            unresolved.append(pid)
            continue
        fid = placement.family.candidates[0]
        family = memory.families[fid]
        if family.kind not in TOWER_KINDS:
            continue
        per_floor, total = family_units(family)
        evidence = (
            SourceRef(memory.snapshot, placement.record.source.pointer + "/hid"),
            SourceRef(memory.snapshot, family.record.source.pointer + "/class_path"),
            SourceRef(
                memory.snapshot, family.record.source.pointer + "/properties/type"
            ),
        )
        towers[pid] = {
            "family_id": fid,
            "family_rule": placement.family.rule,
            "units_per_floor": derived(
                per_floor.value,
                per_floor.state,
                per_floor.unit,
                evidence + per_floor.sources,
                per_floor.rule,
            ),
            "units": derived(
                total.value,
                total.state,
                total.unit,
                evidence + total.sources,
                total.rule,
            ),
        }
    return towers, unresolved

### Descritivos e Grupos Candidatos

`DescritivoTorre` não fornece aqui uma chave explícita para a instância. Usamos o mesmo critério de `descriptor_groups` da aplicação, mas com uma interpretação conservadora: comparar as unidades declaradas no descritivo com as unidades derivadas das famílias instanciadas.

Uma única família compatível produz um **grupo candidato**, que pode conter várias torres. Duas famílias compatíveis produzem ambiguidade. Um descritivo sem correspondência fica sem associação. Se houver instâncias sem família resolvida ou torres sem unidades calculáveis, o resultado é incompleto. A mesma restrição vale quando a lista de instâncias está ausente ou `null`, ou contém tipos de posicionamento desconhecidos. A linha `TOTAL` é um agregado e nunca é associada a uma torre. Nenhuma associação usa ordem de lista ou proximidade do rótulo.

In [5]:
def descriptor_candidates(memory, towers, inventory_complete):
    complete = inventory_complete and all(
        t["units"].state == "known" for t in towers.values()
    )
    descriptors = {}
    for output in memory.solution.outputs:
        if output.data.get("type") != "DescritivoTorre":
            continue
        label = output.data.get("label")
        units = number(output, "/unidades", "unidade habitacional")
        aggregate = isinstance(label, str) and label.strip().upper() == "TOTAL"
        candidates = (
            []
            if aggregate or units.state != "known"
            else [
                pid
                for pid, tower in towers.items()
                if tower["units"].state == "known"
                and tower["units"].value == units.value
            ]
        )
        families = sorted({towers[pid]["family_id"] for pid in candidates})
        status = (
            "aggregate"
            if aggregate
            else units.state
            if units.state != "known"
            else "incomplete"
            if not complete
            else "ambiguous"
            if len(families) > 1
            else "candidate_group"
            if families
            else "unmatched"
        )
        descriptors[output.id] = {
            "source": output.source,
            "label": label,
            "units": units,
            "status": status,
            "family_ids": families,
            "placement_ids": candidates,
            "rule": "same-units-among-instantiated-families/v1",
        }
    return descriptors


def interpret(memory):
    towers, unresolved = tower_semantics(memory)
    inventory_complete = (
        isinstance(lookup(memory.solution.record.data, "/placed_objects"), list)
        and not unresolved
        and all(
            p.kind in {"PlacedFamily", "PlacedElement", "PlacedAuxiliaryArea"}
            for p in memory.solution.placements.values()
        )
    )
    return {
        "filename": memory.filename,
        "snapshot": memory.snapshot,
        "exporter_version": memory.exporter_version,
        "indicators": declared_indicators(memory),
        "towers": towers,
        "unresolved_family_placements": unresolved,
        "placement_inventory_complete": inventory_complete,
        "descriptors": descriptor_candidates(memory, towers, inventory_complete),
    }


DOMAINS = {name: interpret(memory) for name, memory in MEMORIES.items()}
print(f"{len(DOMAINS)} projetos interpretados pelo mesmo contrato.")

37 projetos interpretados pelo mesmo contrato.


## 4. Inspeção de um Projeto

Selecionamos um projeto com torres e descritivos apenas para visualização. A verificação seguinte usa todos os arquivos. Os caminhos abaixo são evidências dentro do snapshot exibido, não identificadores permanentes da aplicação.

In [6]:
EXAMPLE = next(name for name, d in DOMAINS.items() if d["towers"] and d["descriptors"])
example = DOMAINS[EXAMPLE]
display(Markdown(f"**Arquivo:** `{EXAMPLE}`  \n**Snapshot:** `{example['snapshot']}`"))
display(
    pd.DataFrame(
        [
            {
                "conceito": concept,
                "valor": r.value,
                "estado": r.state,
                "unidade": r.unit,
                "natureza": r.nature,
                "origens": [s.pointer for s in r.sources],
            }
            for concept, r in example["indicators"].items()
        ],
        dtype=object,
    )
)
display(
    pd.DataFrame(
        [
            {
                "torre": pid.split("#", 1)[1],
                "familia": t["family_id"].split("#", 1)[1],
                "unidades_pavimento": t["units_per_floor"].value,
                "unidades_torre": t["units"].value,
                "estado": t["units"].state,
            }
            for pid, t in example["towers"].items()
        ],
        dtype=object,
    )
)
display(
    pd.DataFrame(
        [
            {
                "rotulo": d["label"],
                "unidades": d["units"].value,
                "estado": d["status"],
                "familias_candidatas": len(d["family_ids"]),
                "instancias_candidatas": len(d["placement_ids"]),
            }
            for d in example["descriptors"].values()
        ],
        dtype=object,
    )
)
first_tower = next(iter(example["towers"].values()))
display(Markdown("**Evidências do cálculo de unidades da primeira torre:**"))
display(
    pd.DataFrame(
        [
            {"origem": s.pointer, "valor_original": lookup(SOURCES[EXAMPLE], s.pointer)}
            for s in first_tower["units"].sources
        ],
        dtype=object,
    )
)

**Arquivo:** `avaliadas/6ab3be2bc0fe97bead8562f5/bom_6ace404e4a7a894b01852f19.json`  
**Snapshot:** `12a1c0208a27158fba06157614318043c8ead46b2a5f286b68e956f885229ec2`

,conceito,valor,estado,unidade,natureza,origens
0,total_torres,4,known,torre,declared,[/solution/outputs/6/value]
1,total_unidades,2112,known,unidade habitacional,declared,[/solution/outputs/5/value]
2,vagas_projetadas,None,missing,vaga,declared,[/solution]
3,area_construida,88916.62,known,m²,declared,[/solution/outputs/10/value]
4,ca_projetado,2.08,known,adimensional,declared,[/solution/outputs/3/value]


,torre,familia,unidades_pavimento,unidades_torre,estado
0,/solution/placed_objects/1,/families/0,24,528,known
1,/solution/placed_objects/2,/families/0,24,528,known
2,/solution/placed_objects/3,/families/0,24,528,known
3,/solution/placed_objects/4,/families/0,24,528,known


,rotulo,unidades,estado,familias_candidatas,instancias_candidatas
0,01,528,candidate_group,1,4
1,02,528,candidate_group,1,4
2,03,528,candidate_group,1,4
3,04,528,candidate_group,1,4
4,TOTAL,2112,aggregate,0,0


**Evidências do cálculo de unidades da primeira torre:**

,origem,valor_original
0,/solution/placed_objects/1/hid,/building/torre/U/24/3i/23f/bf0ef990-67e4-43c1-9d40-9532576dc8e8
1,/families/0/class_path,/building/torre/U/24/3i/23f
2,/families/0/properties/type,Tower
3,/families/0/properties/pav_tipo/tipologias/~1area~1apartamento~11QV~131_45,24
4,/families/0/properties/n_pavimentos_unidades,22


## 5. Cobertura e Consistência no Acervo

Verificamos que toda evidência existe na fonte correta e que uma leitura declarada conhecida reproduz o valor e o tipo originais. Comparamos também a quantidade de instâncias e a soma das unidades calculadas com os totais declarados. Esses confrontos ficam como `equal`, `different` ou `unavailable`; uma diferença não altera nenhum dos valores.

A concordância é uma verificação de consistência interna. Não comprova a regra de pavimentos, a unidade de área ou a identidade dos descritivos. Campos fora do contrato permanecem na C1 do notebook 2. Os resultados abaixo são calculados nesta execução.

In [7]:
def readings(value):
    if isinstance(value, Reading):
        yield value
    elif isinstance(value, dict):
        for item in value.values():
            yield from readings(item)
    elif isinstance(value, list):
        for item in value:
            yield from readings(item)


def compare(value, declared, available):
    if not available or declared.state != "known":
        return "unavailable"
    return "equal" if value == declared.value else "different"


coverage_rows = []
for name, domain in DOMAINS.items():
    memory = MEMORIES[name]
    for r in readings(domain):
        assert r.sources
        for source in r.sources:
            assert source.snapshot == memory.snapshot
            assert lookup(SOURCES[name], source.pointer) is not MISSING
        if r.nature == "declared" and r.state == "known":
            original = lookup(SOURCES[name], r.sources[0].pointer)
            assert type(r.value) is type(original) and r.value == original
    assert len(domain["descriptors"]) == sum(
        o.data.get("type") == "DescritivoTorre" for o in memory.solution.outputs
    )
    for d in domain["descriptors"].values():
        assert d["source"].snapshot == memory.snapshot
        assert lookup(SOURCES[name], d["source"].pointer) is not MISSING
        assert all(pid in domain["towers"] for pid in d["placement_ids"])
        if d["status"] == "aggregate":
            assert not d["placement_ids"] and not d["family_ids"]
    towers = domain["towers"]
    resolved = domain["placement_inventory_complete"]
    known_units = all(t["units"].state == "known" for t in towers.values())
    total = sum(t["units"].value for t in towers.values()) if known_units else None
    coverage_rows.append(
        {
            "arquivo": name,
            "torres": len(towers),
            "torres_com_unidades": sum(
                t["units"].state == "known" for t in towers.values()
            ),
            "familias_nao_resolvidas": len(domain["unresolved_family_placements"]),
            "total_torres": compare(
                len(towers), domain["indicators"]["total_torres"], resolved
            ),
            "total_unidades": compare(
                total, domain["indicators"]["total_unidades"], resolved and known_units
            ),
        }
    )
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)
display(
    pd.DataFrame(
        {
            concept: Counter(d["indicators"][concept].state for d in DOMAINS.values())
            for concept in CONTRACT
        }
    )
    .fillna(0)
    .astype(int)
    .T
)
descriptor_counts = Counter(
    desc["status"] for d in DOMAINS.values() for desc in d["descriptors"].values()
)
display(
    pd.DataFrame(
        descriptor_counts.items(), columns=["estado_descritivo", "ocorrencias"]
    )
)
display(
    Markdown(
        f"**{len(DOMAINS)} projetos:** "
        "procedência e preservação das leituras verificadas. "
        f"Totais de torres: `{dict(Counter(coverage_df['total_torres']))}`. "
        f"Totais de unidades: `{dict(Counter(coverage_df['total_unidades']))}`."
    )
)

,arquivo,torres,torres_com_unidades,familias_nao_resolvidas,total_torres,total_unidades
0,avaliadas/6ab3be2bc0fe97bead8562f5/bom_6ace404e4a7a894b01852f19.json,4,4,0,equal,equal
1,avaliadas/6ab3be2bc0fe97bead8562f5/otimo_6ace4108cfe0941cdf9a9189.json,12,12,0,equal,equal
2,avaliadas/6ab3be2bc0fe97bead8562f5/ruim_6ace40534a7a894b01852f1d.json,3,3,0,equal,equal
3,avaliadas/6ace3b52f382582e76a9ade0/bom_6ace3c8be748be1a355e6769.json,3,3,0,equal,equal
4,avaliadas/6ace3b52f382582e76a9ade0/otimo_6ace4079bf22db620be742d4.json,2,2,0,equal,equal
...,...,...,...,...,...,...
32,solucoes-18x/6ad294efa4c7ae18390b4e27_6ad2976b402f900729d7a78b.json,2,2,0,equal,equal
33,solucoes-18x/6ad5e108b4865d8d943e406d_6ad5e8c4ef1079329b7e9b0d.json,3,3,0,equal,equal
34,solucoes-18x/6ad60779486e1091c195e188_6ad60861cf7933c2164b0fa5.json,6,6,0,equal,equal
35,solucoes-18x/6ae0eaf96dd7809f17f6a2e3_6ae0ebe491b9dd2b0e000ea9.json,9,9,0,equal,equal


,known,missing
total_torres,37,0
total_unidades,37,0
vagas_projetadas,28,9
area_construida,37,0
ca_projetado,37,0


,estado_descritivo,ocorrencias
0,candidate_group,160
1,aggregate,37


**37 projetos:** procedência e preservação das leituras verificadas. Totais de torres: `{'equal': 37}`. Totais de unidades: `{'equal': 37}`.

## 6. Exportação e Limites

Exportamos o recorte interpretado de um projeto para `experiments/results/`, fora do versionamento. O artefato inclui contrato, versão das regras, estados e procedência. Ele complementa a memória preservada do notebook 2; as referências exigem acesso ao AQG com o mesmo hash. Não é uma exportação completa do domínio nem um contexto pronto para o agente.

Antes de promover esse contrato para a aplicação, precisamos confirmar as unidades e regras com o produtor do AQG e obter identificadores explícitos para os descritivos. Uma futura C2 poderá consultar esses objetos e cálculos, mantendo os estados de ausência e ambiguidade visíveis. A qualidade das respostas do agente ainda exige uma avaliação própria.

In [8]:
EXPORT_DIR = REPO_ROOT / "experiments/results"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_path = EXPORT_DIR / f"domain_{Path(EXAMPLE).stem}.json"
artifact = {
    "schema_version": "aqg-domain-semantics/0.1",
    "unit_basis": UNIT_BASIS,
    "indicator_contract": {
        name: {"output_type": spec[0], "unit": spec[1], "integer": spec[2]}
        for name, spec in CONTRACT.items()
    },
    "domain": example,
}
export_path.write_text(
    json.dumps(artifact, default=asdict, ensure_ascii=False, allow_nan=False, indent=2)
    + "\n",
    encoding="utf-8",
)
print(f"Artefato: {export_path.relative_to(REPO_ROOT)}")
print(f"Tamanho: {export_path.stat().st_size / 1024:.1f} KiB")

Artefato: experiments/results/domain_bom_6ace404e4a7a894b01852f19.json
Tamanho: 18.8 KiB
